# C++ Hybrid Backend: Getting Started, Explained


The demo builds a two-subnetwork hybrid simulation:

- `cortex`: 38 nodes using `JansenRit`, with 1 mode.
- `thalamus`: 38 nodes using `ReducedSetFitzHughNagumo`, with 3 modes.
- one inter-projection from cortex `y1` into thalamus `xi`.

The key API detail is that `compiled.run()` returns a list of `(times, data)` tuples, one tuple per subnetwork. Data has shape `(n_chunks, n_voi, n_nodes, n_modes)`.

## 1. Imports and notebook-safe path setup

The script version can use `__file__` to find its directory. A notebook usually cannot, so this cell searches from the current working directory until it finds `cpp_hybrid_getting_started.py`. It also puts `tvb_library/` on `sys.path`, so local TVB imports resolve when the notebook is run from the repository.

In [ ]:
from __future__ import annotations

import os
import sys
import tempfile
import time
import warnings
from pathlib import Path

def find_examples_dir() -> Path:
    cwd = Path.cwd().resolve()
    candidates = [cwd, cwd / "examples", cwd / "tvb_library/tvb/simulator/backend_cpp/examples"]
    candidates.extend(parent / "examples" for parent in cwd.parents)
    for candidate in candidates:
        if (candidate / "cpp_hybrid_getting_started.py").exists():
            return candidate
    raise RuntimeError("Could not locate backend_cpp/examples directory.")

_EXAMPLES_DIR = find_examples_dir()
_LIBRARY_ROOT = _EXAMPLES_DIR.parent.parent.parent.parent
if str(_LIBRARY_ROOT) not in sys.path:
    sys.path.insert(0, str(_LIBRARY_ROOT))

os.environ.setdefault("TVB_USER_HOME", str(Path(tempfile.gettempdir()) / "tvb-user"))
os.environ.setdefault("MPLCONFIGDIR", str(Path(tempfile.gettempdir()) / "matplotlib"))
os.environ.setdefault("NUMBA_CACHE_DIR", str(Path(tempfile.gettempdir()) / "numba-cache"))

warnings.filterwarnings("ignore", message="Hybrid simulation is experimental.*")

print(f"Examples dir: {_EXAMPLES_DIR}")
print(f"Library root: {_LIBRARY_ROOT}")

## 2. TVB and numerical imports

These are normal TVB hybrid simulator objects. The user-facing model stays Python-level: subnetworks, projections, integrators, monitors, and the `NetworkSet`. The C++ backend later lowers that Python description into generated C++.

In [ ]:
import numpy as np
import scipy.sparse as sp

from tvb.simulator.backend_cpp import CppHybridBackend
from tvb.simulator.hybrid import InterProjection, NetworkSet, Subnetwork
from tvb.simulator.integrators import HeunDeterministic
from tvb.simulator.models import JansenRit, ReducedSetFitzHughNagumo
from tvb.simulator.monitors import TemporalAverage

## 3. Simulation parameters

`DT` is the integration step in milliseconds. `SIMULATION_MS / DT` gives the number of integration steps. `TAVG_PERIOD / DT` gives the monitor chunk size: how many integration steps are averaged into one output sample.

With the default values below, the model runs for 10000 integration steps and produces 1000 temporal-average samples.

In [ ]:
DT = 0.1
N_NODES = 38
SIMULATION_MS = 1000.0
TAVG_PERIOD = 1.0

N_STEP = int(round(SIMULATION_MS / DT))
CHUNK_SIZE = int(round(TAVG_PERIOD / DT))

BUILD_ROOT = Path(
    os.environ.get("TVB_CPP_BUILD_DIR", str(_EXAMPLES_DIR / ".build"))
).resolve()

print(f"N_STEP={N_STEP}, CHUNK_SIZE={CHUNK_SIZE}, BUILD_ROOT={BUILD_ROOT}")

## 4. Build the subnetworks

A `Subnetwork` combines a model, an integration scheme, and a node count. The cortex uses `JansenRit`; the thalamus uses `ReducedSetFitzHughNagumo`. Both use deterministic Heun integration with the same `DT`.

`node_indices` assigns each subnetwork its local node IDs.

In [ ]:
cortex = Subnetwork(
    name="cortex",
    model=JansenRit(),
    scheme=HeunDeterministic(dt=DT),
    nnodes=N_NODES,
).configure()
cortex.node_indices = np.arange(N_NODES)

thalamus = Subnetwork(
    name="thalamus",
    model=ReducedSetFitzHughNagumo(),
    scheme=HeunDeterministic(dt=DT),
    nnodes=N_NODES,
).configure()
thalamus.node_indices = np.arange(N_NODES)

## 5. Inspect model variables

This is not required for execution, but it is useful because projections are specified in terms of source and target variables. The printout shows state variables, variables of interest, coupling variables, and mode counts.

In [ ]:
print("=== Cortex (JansenRit) ===")
print(f"  state_variables: {cortex.model.state_variables}")
print(f"  variables_of_interest: {cortex.model.variables_of_interest}")
print(f"  cvar: {cortex.model.cvar} -> {[cortex.model.state_variables[i] for i in cortex.model.cvar]}")
print(f"  n_modes: {cortex.model.number_of_modes}")

print("\n=== Thalamus (ReducedSetFitzHughNagumo) ===")
print(f"  state_variables: {thalamus.model.state_variables}")
print(f"  variables_of_interest: {thalamus.model.variables_of_interest}")
print(f"  cvar: {thalamus.model.cvar} -> {[thalamus.model.state_variables[i] for i in thalamus.model.cvar]}")
print(f"  n_modes: {thalamus.model.number_of_modes}")

## 6. Define the inter-projection

The projection connects cortex `y1` to thalamus `xi`. The weights are stored as CSR sparse matrices because that is the representation lowered into the native runtime. `lengths_sparse` is all zeros, so this example uses zero-delay coupling.

In [ ]:
rng = np.random.RandomState(42)
weights_dense = np.abs(rng.randn(N_NODES, N_NODES)) * 0.01
weights_sparse = sp.csr_matrix(weights_dense)
lengths_sparse = sp.csr_matrix((N_NODES, N_NODES))

proj = InterProjection(
    source=cortex,
    target=thalamus,
    source_cvar="y1",
    target_cvar="xi",
    weights=weights_sparse,
    lengths=lengths_sparse,
    cv=7.0,
    dt=DT,
    scale=0.1,
)

print("Inter-projection: cortex(y1) -> thalamus(xi)")
print(f"  source_cvar index: {proj.source_cvar}")
print(f"  target_cvar slot: {proj.target_cvar}")

## 7. Assemble and compile the network

`NetworkSet` is the complete hybrid simulation description. `CppHybridBackend.compile()` lowers the Python network to a `SimulationSpec`, renders generated C++ and pybind11 bindings, builds a native extension, and returns a `CompiledCppNetwork` wrapper.

In [ ]:
network = NetworkSet(
    subnets=[cortex, thalamus],
    projections=[proj],
    stimuli=[],
)
network.configure()

monitor = TemporalAverage(period=TAVG_PERIOD)

print(f"Compiling C++ extension to {BUILD_ROOT} ...")
t0 = time.perf_counter()
backend = CppHybridBackend(build_root=BUILD_ROOT)
compiled = backend.compile(
    network,
    monitors=[monitor],
    user_source_hint="cpp_hybrid_getting_started_notebook",
)
compile_time = time.perf_counter() - t0

summary = compiled.debug_summary()
print(f"Compilation done in {compile_time:.1f}s")
print(f"  subnetworks: {summary['n_subnetworks']}")
print(f"  inter-projs: {summary['n_inter_projections']}")
print(f"  intra-projs: {summary['n_intra_projections']}")
print(f"  generated C++: {summary['generated_cpp_path']}")

## 8. Initial conditions

The native backend expects one initial-state array per subnet. Each array has shape `(n_state_variables, n_nodes, n_modes)` and dtype `float64`.

In [ ]:
ic_cortex = np.zeros(
    (cortex.model.nvar, N_NODES, cortex.model.number_of_modes),
    dtype=np.float64,
)
ic_thalamus = np.zeros(
    (thalamus.model.nvar, N_NODES, thalamus.model.number_of_modes),
    dtype=np.float64,
)

print(f"ic_cortex.shape = {ic_cortex.shape}")
print(f"ic_thalamus.shape = {ic_thalamus.shape}")

## 9. Run the compiled simulation

`compiled.run()` calls into the generated extension. It returns one `(times, data)` tuple per subnet, in the same order as `network.subnets`.

The data layout is `(n_chunks, n_voi, n_nodes, n_modes)`. The FHN thalamus keeps its 3 modes in the final axis.

In [ ]:
print(f"Running {SIMULATION_MS} ms ({N_STEP} steps, chunk_size={CHUNK_SIZE}) ...")
t0 = time.perf_counter()
results = compiled.run(
    initial_states=[ic_cortex, ic_thalamus],
    nstep=N_STEP,
    chunk_size=CHUNK_SIZE,
)
run_time = time.perf_counter() - t0

(times_ctx, data_ctx) = results[0]
(times_thal, data_thal) = results[1]

print(f"Simulation done in {run_time:.2f}s")
print(f"Cortex output: times={times_ctx.shape}, data={data_ctx.shape}")
print(f"Thalamus output: times={times_thal.shape}, data={data_thal.shape}")

assert not np.any(np.isnan(data_ctx)), "NaN in cortex output"
assert not np.any(np.isnan(data_thal)), "NaN in thalamus output"
print("No NaNs in output")

## 10. Extract variables of interest

For cortex, VOI index 1 is `y1`, and there is only one mode, so we take `data_ctx[:, 1, :, 0]`.

For thalamus, VOI index 0 is `xi`, but FHN has 3 modes. The C++ backend preserves modes, so the notebook sums over the last axis to match the convention commonly used by the Python/Numba path.

In [ ]:
y1_ctx = data_ctx[:, 1, :, 0]
y1_mean = y1_ctx.mean(axis=1)

xi_thal = data_thal[:, 0, :, :].sum(axis=-1)
xi_mean = xi_thal.mean(axis=1)

print(f"JR y1 amplitude, last 500 ms: {y1_mean[-500:].max() - y1_mean[-500:].min():.4f}")
print(f"FHN xi amplitude, last 500 ms: {xi_mean[-500:].max() - xi_mean[-500:].min():.4f}")

## 11. Optional Numba comparison

This cell runs `NbHybridBackend` when it is available. If Numba rejects `chunk_size=10` for zero-delay projections, the cell runs with `chunk_size=1` and averages back to 1 ms windows before comparing traces.

In [ ]:
def average_chunks(times, data, chunk_size):
    if chunk_size == 1:
        return times, data
    n_window = data.shape[0] // chunk_size
    n_keep = n_window * chunk_size
    if n_window == 0:
        return times, data
    avg_times = times[:n_keep].reshape(n_window, chunk_size).mean(axis=1)
    avg_data = data[:n_keep].reshape(n_window, chunk_size, *data.shape[1:]).mean(axis=1)
    return avg_times, avg_data

nb_t0 = nb_t1 = None
nb_y1_mean = nb_xi_mean = None

try:
    from tvb.simulator.backend.nb_hybrid import NbHybridBackend

    print("Running Numba backend for comparison ...")
    nb_backend = NbHybridBackend()
    try:
        nb_results = nb_backend.run_network(
            network,
            nstep=N_STEP,
            chunk_size=CHUNK_SIZE,
            initial_states=[ic_cortex.copy(), ic_thalamus.copy()],
        )
    except ValueError as exc:
        if "exceeds the minimum projection horizon" not in str(exc):
            raise
        print("Numba rejected chunk_size=10; rerunning chunk_size=1 and averaging.")
        nb_results = nb_backend.run_network(
            network,
            nstep=N_STEP,
            chunk_size=1,
            initial_states=[ic_cortex.copy(), ic_thalamus.copy()],
        )
        nb_results = [(*average_chunks(t, d, CHUNK_SIZE), c) for t, d, c in nb_results]

    nb_t0, nb_d0, _ = nb_results[0]
    nb_t1, nb_d1, _ = nb_results[1]

    nb_y1_mean = np.asarray(nb_d0[:, 1, :, 0], dtype=np.float64).mean(axis=1)
    nb_xi = np.asarray(nb_d1[:, 0, :, :], dtype=np.float64)
    if nb_xi.shape[-1] != 1:
        nb_xi = nb_xi.sum(axis=-1)
    else:
        nb_xi = nb_xi[..., 0]
    nb_xi_mean = nb_xi.mean(axis=1)

    cortex_diff = np.abs(y1_mean - nb_y1_mean).max()
    thalamus_diff = np.abs(xi_mean - nb_xi_mean).max()
    print(f"Cortex max abs C++ - Numba = {cortex_diff:.3e}")
    print(f"Thalamus max abs C++ - Numba = {thalamus_diff:.3e}")
except ImportError:
    print("Numba backend not available; skipping comparison")

## 12. Plot the mean traces

The first panel shows cortex `JansenRit.y1`, averaged over nodes. The second panel shows thalamus FHN `xi`, summed over modes and averaged over nodes. If the Numba comparison cell ran successfully, the dashed black traces show the Numba backend on top of the C++ backend.

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(times_ctx, y1_mean, linewidth=0.8, color="steelblue", label="C++ backend")
if nb_y1_mean is not None:
    axes[0].plot(nb_t0, nb_y1_mean, linewidth=0.8, linestyle="--", color="black", label="Numba backend")
axes[0].set_title("Cortex: JansenRit y1, mean over nodes")
axes[0].set_xlabel("Time (ms)")
axes[0].set_ylabel("y1 (mV)")
axes[0].legend(loc="best")
axes[0].grid(alpha=0.3)

axes[1].plot(times_thal, xi_mean, linewidth=0.8, color="firebrick", label="C++ backend")
if nb_xi_mean is not None:
    axes[1].plot(nb_t1, nb_xi_mean, linewidth=0.8, linestyle="--", color="black", label="Numba backend")
axes[1].set_title("Thalamus: FHN xi, mode-summed")
axes[1].set_xlabel("Time (ms)")
axes[1].set_ylabel("xi (a.u.)")
axes[1].legend(loc="best")
axes[1].grid(alpha=0.3)

fig.suptitle("C++ hybrid backend: cortex (JR) -> thalamus (FHN)", fontsize=13)
plt.tight_layout()
plt.show()

## Summary

The full workflow is:

1. Build Python `Subnetwork` objects.
2. Connect them with an `InterProjection`.
3. Assemble a `NetworkSet`.
4. Compile with `CppHybridBackend`.
5. Run the compiled extension.
6. Unpack one result tuple per subnet.
7. Handle the explicit mode axis in returned C++ data.